# Google WAXAL ASR Challenge - Baseline with Whisper

**Languages:** Lingala (lin), Shona (sna), Luganda (lug)

**Evaluation:** WER (50%) + CER (50%)

**Rules:** Open-source only, no AutoML, set seeds, max 5 submissions/day

In [ ]:
# Install dependencies (run once)
!pip install -q torch torchaudio datasets transformers accelerate evaluate jiwer soundfile librosa pandas numpy tqdm

# For Colab, ensure ffmpeg
import sys
if 'google.colab' in sys.modules:
    !apt-get -qq install ffmpeg

In [ ]:
import os
import random
import numpy as np
import torch
import pandas as pd
from datasets import load_dataset, Audio, Dataset, DatasetDict
from transformers import (
    WhisperFeatureExtractor,
    WhisperTokenizer,
    WhisperProcessor,
    WhisperForConditionalGeneration,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer
)
from evaluate import load as load_metric
import warnings
warnings.filterwarnings('ignore')

# Set seeds for reproducibility
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(42)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {device}')

## 1. Load WAXAL Dataset from HuggingFace

We focus on the 3 target languages: Lingala (lin), Shona (sna), Luganda (lug)

In [ ]:
# Target languages for this challenge
target_languages = {
    'lin': 'Lingala',
    'sna': 'Shona',
    'lug': 'Luganda'
}

dataset_configs = [f'{lang}_asr' for lang in target_languages.keys()]
print(f'Loading configs: {dataset_configs}')

# Load each language's ASR dataset from HuggingFace
datasets_dict = {}
for config in dataset_configs:
    print(f'Loading {config}...')
    ds = load_dataset('google/WaxalNLP', config, trust_remote_code=True)
    datasets_dict[config] = ds
    print(f'  {config}: train={len(ds["train"])}, val={len(ds["validation"])}, test={len(ds["test"])}')

In [ ]:
# Combine all languages into a single dataset
def combine_splits(datasets_dict, split='train'):
    combined = None
    for config, ds in datasets_dict.items():
        lang = config.split('_')[0]
        split_ds = ds[split]
        # Add language column
        split_ds = split_ds.map(lambda x, lang=lang: {'language_code': lang})
        if combined is None:
            combined = split_ds
        else:
            combined = concatenate_datasets([combined, split_ds])
    return combined

from datasets import concatenate_datasets

# Create combined dataset
combined_dataset = DatasetDict({
    'train': combine_splits(datasets_dict, 'train'),
    'validation': combine_splits(datasets_dict, 'validation'),
    'test': combine_splits(datasets_dict, 'test'),
})

print(f'Combined: train={len(combined_dataset["train"])}, val={len(combined_dataset["validation"])}, test={len(combined_dataset["test"])}')
print(f'Sample: {combined_dataset["train"][0]["transcription"][:100]}')

## 2. Load Whisper Model

Using Whisper-small as baseline (can be swapped with base/medium/large)

In [ ]:
model_name = 'openai/whisper-small'

# Load processor, feature extractor, tokenizer
feature_extractor = WhisperFeatureExtractor.from_pretrained(model_name)
tokenizer = WhisperTokenizer.from_pretrained(model_name, language='en', task='transcribe')
processor = WhisperProcessor.from_pretrained(model_name, language='en', task='transcribe')

# Load model
model = WhisperForConditionalGeneration.from_pretrained(model_name)
model.config.forced_decoder_ids = None  # Allow multilingual
model.config.suppress_tokens = []

# Resample to 16kHz
combined_dataset = combined_dataset.cast_column('audio', Audio(sampling_rate=16000))

print(f'Model parameters: {model.num_parameters():,}')

In [ ]:
# Prepare dataset for training - map audio to input features
def prepare_dataset(batch):
    audio = batch['audio']
    batch['input_features'] = feature_extractor(
        audio['array'], sampling_rate=audio['sampling_rate']
    ).input_features[0]
    batch['labels'] = tokenizer(batch['transcription']).input_ids
    return batch

# Only take a subset for faster prototyping (can remove for full training)
train_sample = combined_dataset['train'].shuffle(seed=42).select(range(min(1000, len(combined_dataset['train']))))
val_sample = combined_dataset['validation'].shuffle(seed=42).select(range(min(200, len(combined_dataset['validation']))))

# Apply preprocessing
train_dataset = train_sample.map(prepare_dataset, remove_columns=['audio', 'transcription', 'speaker_id', 'gender', 'id', 'language', 'language_code'])
val_dataset = val_sample.map(prepare_dataset, remove_columns=['audio', 'transcription', 'speaker_id', 'gender', 'id', 'language', 'language_code'])

print(f'Processed train: {len(train_dataset)}, val: {len(val_dataset)}')

In [ ]:
# Data collator for Whisper
from transformers import Seq2SeqDataCollator

data_collator = Seq2SeqDataCollator(
    processor=processor,
    model=model
)

In [ ]:
# Compute metrics function
wer_metric = load_metric('wer')
cer_metric = load_metric('cer')

def compute_metrics(pred):
    pred_ids = pred.predictions
    label_ids = pred.label_ids

    # Replace -100 with pad token
    label_ids[label_ids == -100] = tokenizer.pad_token_id

    pred_str = tokenizer.batch_decode(pred_ids, skip_special_tokens=True)
    label_str = tokenizer.batch_decode(label_ids, skip_special_tokens=True)

    wer = wer_metric.compute(predictions=pred_str, references=label_str)
    cer = cer_metric.compute(predictions=pred_str, references=label_str)

    # Combined score: 0.5 * WER + 0.5 * CER
    combined = 0.5 * wer + 0.5 * cer

    return {'wer': wer, 'cer': cer, 'combined': combined}

In [ ]:
# Training arguments - adjusted for Colab free tier
training_args = Seq2SeqTrainingArguments(
    output_dir='./whisper-waxal-results',
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=2,
    learning_rate=1e-5,
    warmup_steps=100,
    num_train_epochs=3,
    evaluation_strategy='steps',
    eval_steps=100,
    save_steps=100,
    logging_steps=25,
    report_to=['none'],
    predict_with_generate=True,
    generation_max_length=225,
    fp16=True,
    save_total_limit=2,
    seed=42,
    data_seed=42,
    load_best_model_at_end=True,
    metric_for_best_model='combined',
    greater_is_better=False,
)

In [ ]:
# Create trainer
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    tokenizer=processor.feature_extractor,
)

In [ ]:
# Train the model
trainer.train()

# Save best model
trainer.save_model('./whisper-waxal-best')

## 3. Generate Test Predictions

Run inference on the test set and create submission file.

In [ ]:
# Load best model for inference
best_model = WhisperForConditionalGeneration.from_pretrained('./whisper-waxal-best')
best_model.to(device)
best_model.eval()

# Process test set
test_dataset = combined_dataset['test']

def transcribe_batch(batch):
    audio = batch['audio']
    inputs = feature_extractor(
        audio['array'], sampling_rate=audio['sampling_rate'],
        return_tensors='pt'
    ).input_features.to(device)

    with torch.no_grad():
        predicted_ids = best_model.generate(inputs)

    transcription = processor.batch_decode(predicted_ids, skip_special_tokens=True)[0]
    return {'prediction': transcription}

# Apply to test set (may take a while)
predictions = test_dataset.map(transcribe_batch)

# Create submission dataframe
submission_df = pd.DataFrame({
    'id': predictions['id'],
    'transcription': predictions['prediction']
})

submission_df.to_csv('../submissions/baseline_whisper_submission.csv', index=False)
print(f'Submission saved! Shape: {submission_df.shape}')
print(submission_df.head())

## 4. Evaluate on Validation Set

In [ ]:
# Evaluate on validation set
val_predictions = combined_dataset['validation'].map(transcribe_batch)

references = val_predictions['transcription']
hypotheses = val_predictions['prediction']

wer = wer_metric.compute(predictions=hypotheses, references=references)
cer = cer_metric.compute(predictions=hypotheses, references=references)
combined = 0.5 * wer + 0.5 * cer

print(f'Validation Results:')
print(f'  WER: {wer:.4f}')
print(f'  CER: {cer:.4f}')
print(f'  Combined: {combined:.4f}')